In [1]:
%%bash
cat << 'EOF' > /content/config.sh
#!/bin/bash

export ROOTDIR="/content/work"
export VIDEOSOURCE="gsplat/input/IMG_4765.MOV"
export IMAGESET="gsplat/input/perfume/images"
export INPUT_MODE="images"

export FPS=25
#export PROFILE="gpu/quality"
export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"
EOF

In [2]:
!cat /content/config.sh

#!/bin/bash

export ROOTDIR="/content/work"
export VIDEOSOURCE="gsplat/input/IMG_4765.MOV"
export IMAGESET="gsplat/input/perfume/images"
export INPUT_MODE="images"

export FPS=25
#export PROFILE="gpu/quality"
export PROFILE="gpu/balanced"
#export PROFILE="cpu/fast"


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda
!/usr/local/miniconda/bin/conda init bash

PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda
no change     /usr/local/miniconda/condabin/conda
no change     /usr/local/miniconda/bin/conda
no change     /usr/local/miniconda/bin/conda-env
no change     /usr/local/miniconda/bin/activate
no change     /usr/local/miniconda/bin/deactivate
no change     /usr/local/miniconda/etc/profile.d/conda.sh
no change     /usr/local/miniconda/etc/fish/conf.d/conda.fish
no change     /usr/local/miniconda/shell/condabin/Conda.psm1
no change     /usr/loc

In [36]:
%%bash
set -e
source /content/config.sh

mkdir -p "$ROOTDIR"

if [ -n "$VIDEOSOURCE" ]; then
  SRC_VIDEO="/content/drive/MyDrive/$VIDEOSOURCE"

  if [ ! -f "$SRC_VIDEO" ]; then
    echo "❌ Video not found: $SRC_VIDEO"
  else
    echo "🎬 Copying video: $SRC_VIDEO"
    cp -f "$SRC_VIDEO" "$ROOTDIR/video.mp4"
  fi
fi

if [ -n "$IMAGESET" ]; then
  SRC_IMAGES="/content/drive/MyDrive/$IMAGESET"
  DST_IMAGES="$ROOTDIR/images"

  if [ ! -d "$SRC_IMAGES" ]; then
    echo "❌ Image dataset not found: $SRC_IMAGES"
  else
    echo "🖼️ Preparing image dataset: $SRC_IMAGES"

    mkdir -p "$DST_IMAGES"

    COUNT=$(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l | tr -d ' ')
    if [ "$COUNT" -lt 2 ]; then
      echo "❌ Not enough images ($COUNT)"
      exit 1
    fi

    rm -f "$DST_IMAGES"/frame_*.png 2>/dev/null || true

    i=1
    for img in $(find "$SRC_IMAGES" -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | sort); do
      printf -v idx "%05d" "$i"
      cp "$img" "$DST_IMAGES/frame_${idx}.png"
      i=$((i+1))
    done

    echo "✅ Dataset ready in $DST_IMAGES"
  fi
fi

❌ Video not found: /content/drive/MyDrive/gsplat/input/IMG_4765.MOV
🖼️ Preparing image dataset: /content/drive/MyDrive/gsplat/input/perfume/images
✅ Dataset ready in /content/work/images


In [37]:
rm -rf /content/images

In [23]:
!git clone https://github.com/NicoIGN/video_to_ply.git
%cd video_to_ply

Cloning into 'video_to_ply'...
remote: Enumerating objects: 771, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 771 (delta 3), reused 9 (delta 3), pack-reused 760 (from 1)
Receiving objects: 100% (771/771), 423.45 KiB | 11.14 MiB/s, done.
Resolving deltas: 100% (490/490), done.
/content/video_to_ply/video_to_ply


In [24]:
%%bash
cd /content/video_to_ply
git stash save && git checkout dev && git pull

No local changes to save
Your branch is up to date with 'origin/dev'.
Already up to date.


Already on 'dev'


In [25]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


#conda env update -n gsplat -f environment/conda_colab.yml --prune
conda env create -n gsplat -f environment/conda_colab.yml


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: / - \ | / - done
Installing pip dependencies: | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / Ran pip subprocess with arguments:
['/usr/local/miniconda/envs/gsplat/bin/python', '-m', 'pip', 'install', '-U', '-r', '/content/video_to_ply/video_to_ply/environment/condaenv.y88vyr7c.requirements.txt', '--exists-action=b']
Pip subprocess output:
  Using cached torchvision-0.26.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a whi



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda




In [ ]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && \
source /content/config.sh && \
echo INPUT_MODE=$INPUT_MODE && \
cd /content/video_to_ply/ && \
conda activate gsplat && \
INPUT_ARG="" && \
if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
  INPUT_ARG="--images $ROOTDIR/images"; \
elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
  INPUT_ARG="--video $ROOTDIR/video.mp4 --fps $FPS"; \
fi && \
bash run.sh $INPUT_ARG --root "$ROOTDIR" --skip-conda --profile "$PROFILE" --no-proxy

INPUT_MODE=images
🚫 Proxy disabled (NO_PROXY=true)
👉 using profile: gpu/balanced
⏩ Skipping conda setup (--skip-conda enabled)
✅ Using python: Python 3.10.20 
🚀 GPU model OK: splatfacto
🖼️ Importing images from: /content/work/images
📥 Copying 126 images to /content/work/input
📦 ROOT: /content/work
🖼️ Preparing images → /content/work/ori/images
🚫 Proxy disabled (NO_PROXY=true)
🧹 Preparing output directory: /content/work/ori/images
🖼️ Preparing images from: /content/work/images
📁 Output: /content/work/ori/images
✅ Prepared 126 images
🧼 Removing non-frame files from output directory...
✅ Output directory clean (only frame_*.png present)


🧭 Running COLMAP through NerfStudio...
🚫 Proxy disabled (NO_PROXY=true)
────────────────────────────────────
📁 INPUT   : /content/work/ori/images
📁 OUTPUT  : /content/work/ori
⚙️ DEVICE  : gpu
────────────────────────────────────
🚀 Running ns-process-data...
✅ ns-process-data SUCCESS
📦 OUTPUT READY: /content/work/ori


🧠 Training...
🚫 Proxy disabled (NO_

In [11]:
!source /usr/local/miniconda/etc/profile.d/conda.sh && \
source /content/config.sh && \
echo "skipping all except exporting step" && \
cd /content/video_to_ply/ && \
conda activate gsplat && \
INPUT_ARG="" && \
if [ "$INPUT_MODE" = "images" ] && [ -n "$IMAGESET" ]; then \
  INPUT_ARG="--images $ROOTDIR/images"; \
elif [ "$INPUT_MODE" = "video" ] && [ -n "$VIDEOSOURCE" ]; then \
  INPUT_ARG="--video $ROOTDIR/video.mp4 --fps $FPS"; \
fi && \
bash run.sh $INPUT_ARG --root "$ROOTDIR" --skip-conda --profile "$PROFILE" --no-proxy --skip-conda --skip-frame-extraction --skip-colmap  --skip-training

skipping all steps except exporting step
🚫 Proxy disabled (NO_PROXY=true)
❌ Images directory not found: /content/gsplat/input/perfume/images


In [12]:
from google.colab import files
import os
import subprocess
import glob

# =========================
# LOAD CONFIG.SH VARIABLES
# =========================
result = subprocess.run(
    "source /content/config.sh && env",
    shell=True,
    executable="/bin/bash",
    capture_output=True,
    text=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

# =========================
# FIND ALL PLY FILES
# =========================
rootdir = os.environ.get("ROOTDIR", "/content/work")
export_dir = os.path.join(rootdir, "exports")

ply_files = sorted(
    glob.glob(os.path.join(export_dir, "*.ply"))
)

if not ply_files:
    print("⚠️ No PLY files found.")
    print(f"📂 Searched in: {export_dir}")
else:
    print(f"📦 Found {len(ply_files)} PLY file(s):")

    for ply_path in ply_files:
        print(f"⬇️ Downloading: {os.path.basename(ply_path)}")
        files.download(ply_path)

⚠️ No PLY files found.
📂 Searched in: /content/work/exports
